<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/Uba_fiuba_ingenieria_logo.png" width="300" align="center">

# **Análisis de Series de Tiempo II**
# **Clase 8, Pipelines Temporales y Serving**



El modelo es el 5% de un sistema de forecasting. El otro 95% es plomería temporal: que los datos lleguen, que las features se calculen igual en todos lados, y que el resultado se publique de forma auditable.

Secciones:

1. **El reloj de producción**: el sistema solo ve lo que existía hasta el corte.
2. **Una sola función de features**: la regla que evita un bug clave de ML en producción.
3. **El contrato del pronóstico**: que fila publica el sistema.
4. **Training/serving skew, medido**: problema que puede estar presente

**Dataset:** ETTh1, carga horaria de un transformador eléctrico.

Importamos lo necesario

In [ ]:
import time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor

warnings.filterwarnings("ignore")

Cargamos el dataset

In [ ]:
URL = "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"
raw = pd.read_csv(URL, parse_dates=["date"]).rename(columns={"date": "timestamp"})

Definimosa la tabla que contiene la serie a pronosticar (y) y una covariable (cov)

In [2]:
tabla = raw[["timestamp", "HUFL", "LUFL"]].rename(columns={"HUFL": "y", "LUFL": "cov"})

HORIZONTES = [1, 12, 24] # pronosticamos a 1, 12 y 24 horas
QUANTILES  = [0.1, 0.5, 0.9] # pronóstico probabilístico, no solo la media

print(tabla.shape, "|", tabla.timestamp.min(), "→", tabla.timestamp.max())
tabla.head(3)

(17420, 3) | 2016-07-01 00:00:00 → 2018-06-26 19:00:00


,timestamp,y,cov
0,2016-07-01 00:00:00,5.827,4.203
1,2016-07-01 01:00:00,5.693,4.142
2,2016-07-01 02:00:00,5.157,3.777


## A) El reloj de producción

* En producción el sistema no ve el dataset: ve lo que existía hasta un instante de corte.

* Toda la lógica pasa por esta función, y nadie vuelve a tocar tabla directamente. Así el leakage se vuelve estructuralmente imposible en vez de depender de que uno se acuerde.

In [3]:
def historia_al_corte(corte):
    #Lo único que el sistema puede ver en el instante 'corte'
    return tabla.loc[tabla.timestamp <= corte]

CORTE = pd.Timestamp("2018-05-01 23:00:00")  # el "hoy" de nuestros ejemplos

el sistema ve 16080 filas
y debe pronosticar de acá en adelante


El sistema ve 16080 filas y debe pronosticar de acá en adelante

## B) Una sola función de features


* Si el código que arma las features para entrenar no es literalmente el mismo que el que las arma para predecir, tarde o temprano va a dejar de coincidir. A eso se lo llama training/serving skew, y en la sección D vamos a medir cuánto cuesta.

* La función recibe la historia disponible y el instante a predecir, y devuelve una fila de features.


Dos detalles que hay que tener en cuenta:

- el calendario se calcula sobre el instante a predecir
- los lags y promedios se calculan desde el corte (nunca desde el futuro).

In [4]:
def construir_features(hist, target_time):
    """Features para prede  cir 'target_time' usando solo 'hist'.

    Es la unica fuente de verdad de features del sistema: la usan por igual
    el job de entrenamiento y el de inferencia.
    """
    y = hist["y"].to_numpy()
    corte = hist["timestamp"].iloc[-1]

    return {
        # calendario del instante a predecir (se conoce de antemano)
        "hora": target_time.hour,
        "dia_sem": target_time.dayofweek,
        "hora_sin": np.sin(2*np.pi*target_time.hour/24), # las 23h y las 0h quedan cerca
        "hora_cos": np.cos(2*np.pi*target_time.hour/24),

        # cuán lejos estamos mirando: el modelo tiene que saberlo
        "horizonte": int((target_time - corte) / pd.Timedelta(1, "h")),

        # historia reciente, medida desde el corte
        "lag_1": y[-1],
        "lag_24": y[-24],
        "lag_168": y[-168],
        "media_24": y[-24:].mean(),
        "std_24": y[-24:].std(),
        "media_168": y[-168:].mean(),

        # covariable: solo valores pasados (su futuro no lo conocemos)
        "cov_lag_1":  hist["cov"].to_numpy()[-1],
    }

COLUMNAS = list(construir_features(historia_al_corte(CORTE), CORTE + pd.Timedelta(1, "h")))
print(f"{len(COLUMNAS)} features:", COLUMNAS)

12 features: ['hora', 'dia_sem', 'hora_sin', 'hora_cos', 'horizonte', 'lag_1', 'lag_24', 'lag_168', 'media_24', 'std_24', 'media_168', 'cov_lag_1']


### El dataset de entrenamiento sale de la misma función

Para armar cada fila de entrenamiento simulamos un corte pasado: nos paramos en un día viejo y
preguntamos "¿qué habría visto el sistema ese día?". Es más lento que un shift() vectorizado, pero es
correcto por construcción.

Entrenamos un modelo por horizonte y por cuantil (2 × 3 = 6 modelos). Es la estrategia direct
multi-horizon: cada modelo aprende a predecir "dentro de h horas" y no acumula error como el enfoque
recursivo.

In [ ]:
FIN_TRAIN = pd.Timestamp("2018-04-01")

Tomar atención a la frecuencia de los cortes:
* si tomamos uno cada 12 horas, el modelo solo ve dos horas del día en el entrenamiento y nunca aprende el ciclo diario
* con 7h el corte va rotando por todas las horas

In [5]:
cortes_train = pd.date_range("2016-08-01", FIN_TRAIN, freq="7h")

def armar_dataset(cortes, h):
    # (X, y) para el modelo del horizonte h
    valores = tabla.set_index("timestamp")["y"]
    filas, targets = [], []
    for corte in cortes:
        hist = historia_al_corte(corte)
        objetivo = corte + pd.Timedelta(h, "h")
        if len(hist) < 200 or objetivo not in valores.index:
            continue
        filas.append(construir_features(hist, objetivo))
        targets.append(valores.loc[objetivo])
    return pd.DataFrame(filas)[COLUMNAS], np.array(targets)

t0 = time.time()
modelos = {}
for h in HORIZONTES:
    X, y = armar_dataset(cortes_train, h)
    for q in QUANTILES:
        modelos[(h, q)] = HistGradientBoostingRegressor(
            loss="quantile", quantile=q, max_iter=150, random_state=0).fit(X, y)
    print(f"h={h:2d}: {len(X)} filas × {X.shape[1]} features")

print(f"\n{len(modelos)} modelos entrenados en {time.time()-t0:.0f}s")

h= 1: 2085 filas × 12 features
h=12: 2085 filas × 12 features
h=24: 2085 filas × 12 features

9 modelos entrenados en 9s


## C) El contrato del pronóstico

El sistema no deberia darnos un numero, si no una fila auditable.

| Campo | Para que sirve |
|---|---|
| `forecast_time` | cuándo se generó, sin esto no se puede monitorear nada |
| `target_time` | para qué momento aplica |
| `horizon` | a cuántos pasos |
| `q10 / q50 / q90` | la distribución, no solo la media |
| `model_version` | quién lo produjo, habilita auditoría y rollback |

Y un detalle importante, como cada cuantil se entrena por separado, nada garantiza que q10 ≤ q50 ≤ q90. Si se cruzan, el intervalo queda invertido y rompe cualquier decisión, la
corrección es ordenarlos.

In [6]:
def job_inferencia(corte, version="hgb-v1"):
    # Genera el pronóstico publicable para un corte dado
    hist = historia_al_corte(corte)  # la única entrada al sistema
    filas = []
    for h in HORIZONTES:
        objetivo = corte + pd.Timedelta(h, "h")
        # misma funcion de features que en entrenamiento
        x = pd.DataFrame([construir_features(hist, objetivo)])[COLUMNAS]
        fila = {"forecast_time": corte, "target_time": objetivo, "horizon": h}
        for q in QUANTILES:
            fila[f"q{int(q*100)}"] = float(modelos[(h, q)].predict(x)[0])
        filas.append(fila)

    out = pd.DataFrame(filas)
    out[["q10","q50","q90"]] = np.sort(out[["q10","q50","q90"]].to_numpy(), axis=1)  # evita el cruce
    out["model_version"] = version
    return out

job_inferencia(CORTE)

,forecast_time,target_time,horizon,q10,q50,q90,model_version
0,2018-05-01 23:00:00,2018-05-02 00:00:00,1,13.545062,15.053658,16.318126,hgb-v1
1,2018-05-01 23:00:00,2018-05-02 11:00:00,12,-4.873351,0.463508,6.809355,hgb-v1
2,2018-05-01 23:00:00,2018-05-02 23:00:00,24,11.774550,13.942003,16.643634,hgb-v1


### Baseline
Corremos 40 días fuera del período de entrenamiento y comparamos contra el **naive estacional**
(lo que pasó hace exactamente una semana).

In [7]:
cortes_test = pd.date_range("2018-04-05 23:00:00", periods=40, freq="24h")
reales = tabla.set_index("timestamp")["y"]

pron = pd.concat([job_inferencia(c_) for c_ in cortes_test], ignore_index=True)
ev = pron.merge(reales.rename("real"), left_on="target_time", right_index=True)

ev["error"]    = (ev.real - ev.q50).abs()
ev["naive"]    = [reales.get(t - pd.Timedelta(168, "h")) for t in ev.target_time]
ev["error_nv"] = (ev.real - ev.naive).abs()
ev["dentro"]   = (ev.real >= ev.q10) & (ev.real <= ev.q90)

print(ev.groupby("horizon")[["error", "error_nv", "dentro"]].mean().round(2), "\n")
print(f"MAE modelo: {ev.error.mean():.2f} | naive: {ev.error_nv.mean():.2f} "
      f"| mejora: {1 - ev.error.mean()/ev.error_nv.mean():+.0%}")
print(f"cobertura del intervalo 80%: {ev.dentro.mean():.0%}  (debería dar ~80%)")

         error  error_nv  dentro
horizon                         
1         1.27      2.13    0.75
12        7.62     10.45    0.48
24        1.87      2.65    0.68 

MAE modelo: 3.58 | naive: 5.08 | mejora: +29%
cobertura del intervalo 80%: 63%  (debería dar ~80%)


## D) Training/serving skew, medido

* El bug más caro de ML en producción es que las features se calculan de una manera al entrenar y de otra al servir.

* No falla, solo predice peor.

* Escribimos dos versiones "razonables pero distintas" de la función de features y medimos el daño.

In [8]:
def features_skew_1(hist, target_time):
    """Skew 1: en producción llega una hora menos de historia (off-by-one del ETL)."""
    return construir_features(hist.iloc[:-1], target_time)

def features_skew_2(hist, target_time):
    """Skew 2: el calendario se calcula sobre el CORTE y no sobre el instante a predecir.
    Es el error más común en multi-horizonte: 'hora' termina siendo siempre la del corte."""
    f = construir_features(hist, target_time)
    corte = hist.timestamp.iloc[-1]
    f["hora"] = corte.hour
    f["hora_sin"] = np.sin(2*np.pi*corte.hour/24)
    f["hora_cos"] = np.cos(2*np.pi*corte.hour/24)
    return f

def mae_con(fn_features):
    """Sirve los mismos cortes usando otra función de features y devuelve el MAE."""
    filas = []
    for corte in cortes_test:
        hist = historia_al_corte(corte)
        for h in HORIZONTES:
            objetivo = corte + pd.Timedelta(h, "h")
            x = pd.DataFrame([fn_features(hist, objetivo)])[COLUMNAS]
            filas.append({"target_time": objetivo,
                          "q50": float(modelos[(h, 0.5)].predict(x)[0])})
    d = pd.DataFrame(filas).merge(reales.rename("real"), left_on="target_time", right_index=True)
    return (d.real - d.q50).abs().mean()

base = mae_con(construir_features)
print(f"features correctas          : MAE {base:.2f}   (referencia)")
for nombre, fn in [("SKEW 1 (una hora menos)", features_skew_1),
                   ("SKEW 2 (calendario del corte)", features_skew_2)]:
    m = mae_con(fn)
    print(f"{nombre:28s}: MAE {m:.2f}   ({m/base - 1:+.0%})")

features correctas          : MAE 3.58   (referencia)
SKEW 1 (una hora menos)     : MAE 5.03   (+40%)
SKEW 2 (calendario del corte): MAE 6.76   (+89%)
